# Week 5 — Semantic Search with Embeddings
### Implementation, relevance judgments, latency benchmarking, and conclusions

**Project:** IDX real-estate listing search  
**Evaluation corpus:** 10,000 distinct real listing IDs (final benchmark)  
**Retriever:** `all-MiniLM-L6-v2` sentence embeddings + FAISS; **baseline:** BM25  
**Evaluation date:** October 2026

> **Result:** On the final 100-query benchmark, semantic search reached **10.848 ms mean** and **11.397 ms p95**—both below the 100 ms threshold. The earlier 10-query benchmark had unstable tail latency. Values below are supplied benchmark measurements, not freshly measured by this notebook.

This is a report notebook; its core tables and plots run without downloading a model or rebuilding an index. An optional section provides commands to reproduce the benchmark in the project environment.

## 1. Goals and deliverables

| Week 5 requirement | Outcome / qualification |
|---|---|
| Sentence embeddings, 384 or 768 dimensions | **384-dimensional** MiniLM embeddings implemented; final report confirms dimension |
| Efficient FAISS index | **FAISS inner-product** index with L2-normalized vectors (cosine similarity) |
| `SemanticSearcher` | Implemented in `scripts/semantic_search.py` |
| BM25 baseline | Implemented and benchmarked against same-sized corpus |
| Latency under 100 ms at 10k | **Passed** at both mean and p95 in final 100-run benchmark |
| 50 query–result relevance judgments | **50 pairs labeled** (grades 0/1/2); not sufficient alone for unbiased ranking metrics |
| Precision / recall / NDCG comparison | **Not established** from one judged result per query; needs pooled top-k judging |

The stated Week 5 work involves offline embedding/index construction and online query-time ranking; those times are reported separately.

In [ ]:
import math
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

REPORTS = {
    '1k_previous': {
        'listing_count': 1000,
        'timed_queries': 30,
        'semantic': {'mean': 28.25, 'p50': 6.87, 'p95': 29.10},
        'bm25': {'mean': 0.56, 'p50': 0.63, 'p95': 0.71},
    },
    '10k_initial': {
        'listing_count': 10000,
        'timed_queries': 10, 'warmup': 2,
        'model_load_s': 8.594, 'faiss_build_s': 123.974, 'bm25_build_s': 1.040,
        'semantic': {'mean': 30.945, 'p50': 9.835, 'p95': 125.574, 'p99': 196.272, 'max': 213.946},
        'bm25': {'mean': 15.072, 'p50': 15.433, 'p95': 17.536, 'p99': 17.825, 'max': 17.897},
    },
    '10k_final': {
        'listing_count': 10000, 'timed_queries': 100, 'warmup': 10,
        'model_load_s': 8.844, 'faiss_build_s': 136.250, 'bm25_build_s': 1.054,
        'semantic': {'mean': 10.848, 'p50': 11.001, 'p95': 11.397, 'p99': 11.612, 'max': 12.048},
        'bm25': {'mean': 14.686, 'p50': 14.868, 'p95': 18.352, 'p99': 20.089, 'max': 20.676},
    }
}
TARGET_MS = 100.0
final = REPORTS['10k_final']
print('Final corpus:', f"{final['listing_count']:,}", 'distinct listing IDs')
print('Embedding dimensions:', 384)
print('Timed queries per engine:', final['timed_queries'])
print('Mean threshold passed:', final['semantic']['mean'] < TARGET_MS)
print('p95 threshold passed:', final['semantic']['p95'] < TARGET_MS)

## 2. Search architecture and ranking

**Semantic retrieval:** Listing remark → SentenceTransformer embedding (384-D) → float32 L2 normalization → FAISS `IndexFlatIP` → ranked top-10 results. At query time, encode the query using the same model, normalize, and search the index. The normalized inner product ranks by cosine similarity.

**BM25 baseline:** Tokenize all listing remarks into a lexical index. At query time, tokenize the query, compute BM25 scores, and return the highest-ranking documents. BM25 does not require neural inference for each query.

Both engines operate over listing text and should return stable listing IDs. **Neither engine automatically guarantees that structured constraints such as exact prices, bedroom counts, or listing status are enforced.** Those requirements belong in the structured SQL-filter stage or a validated hybrid retrieval pipeline.

In [ ]:
methods = pd.DataFrame([
    {'Aspect': 'Representation', 'Semantic / FAISS': '384-D dense vectors', 'BM25': 'Token-level lexical statistics'},
    {'Aspect': 'Similarity', 'Semantic / FAISS': 'Normalized inner product (cosine)', 'BM25': 'Term-based BM25 relevance score'},
    {'Aspect': 'Indexing', 'Semantic / FAISS': 'Embedding generation + FAISS', 'BM25': 'Tokenization + BM25 corpus statistics'},
    {'Aspect': 'Query work', 'Semantic / FAISS': 'Encode query + vector search', 'BM25': 'Tokenize query + score documents'},
    {'Aspect': 'Strength', 'Semantic / FAISS': 'Semantic paraphrases and related wording', 'BM25': 'Exact terms and explicit keywords'},
])
display(methods)

## 3. Benchmark design

- **Data:** `data/processed/listing_sample_10k.csv`, containing 10,000 distinct real listing IDs.
- **Embeddings:** 384-D MiniLM representations for listing remarks.
- **Top-k:** 10 retrieved listings per query.
- **Warm-up:** 10 queries per engine on the final run.
- **Timed calls:** 100 queries per engine on the final run; 10 on the preliminary 10k run.
- **Measured scope:** Warm end-to-end `engine.search()` calls (query encoding/tokenization, retrieval, and result construction).
- **Excluded from latency:** CSV reading, model loading, offline indexing.
- **Hardware context:** User-reported Mac environment; exact CPU/memory configuration and background load were not captured in the supplied metrics.

The reports contain aggregate quantiles, **not individual per-query timings**. Accordingly, percentile graphics below use provided summaries and do not pretend to show a raw distribution or confidence interval.

## 4. Final 10k latency results (100 timed queries)

In [ ]:
latency = pd.DataFrame([
    {'Engine': engine, 'Mean (ms)': vals['mean'], 'p50 (ms)': vals['p50'],
     'p95 (ms)': vals['p95'], 'p99 (ms)': vals['p99'], 'Max (ms)': vals['max'],
     'Timed queries': final['timed_queries']}
    for engine, vals in [('Semantic (FAISS)', final['semantic']), ('BM25', final['bm25'])]
])
display(latency)
print(f"Semantic mean < {TARGET_MS:.0f} ms: {final['semantic']['mean'] < TARGET_MS}")
print(f"Semantic p95 < {TARGET_MS:.0f} ms: {final['semantic']['p95'] < TARGET_MS}")
print(f"Semantic p95 headroom: {TARGET_MS - final['semantic']['p95']:.3f} ms")
print(f"Semantic mean vs BM25 mean: {final['bm25']['mean']/final['semantic']['mean']:.2f}x (BM25 / semantic)")

In [ ]:
metric_keys = [('mean','Mean'), ('p50','p50'), ('p95','p95'), ('p99','p99'), ('max','Max')]
x = np.arange(len(metric_keys)); width = 0.36
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x-width/2, [final['semantic'][k] for k,_ in metric_keys], width, label='Semantic / FAISS')
ax.bar(x+width/2, [final['bm25'][k] for k,_ in metric_keys], width, label='BM25')
ax.axhline(TARGET_MS, linestyle='--', linewidth=1, label='100 ms target')
ax.set(xticks=x, xticklabels=[name for _,name in metric_keys], ylabel='Latency (ms)',
       title='Final 10,000-listing latency: 100 timed searches per engine')
ax.legend(); ax.grid(axis='y', alpha=0.2)
plt.tight_layout(); plt.show()

**Interpretation:** The final run meets the original **mean under 100 ms** goal and also meets the stronger **p95 under 100 ms** criterion. In this benchmark, semantic mean latency (10.848 ms) was lower than BM25 mean latency (14.686 ms). This result is specific to the reported dataset, implementation, hardware conditions and timing scope—not a universal claim that neural retrieval is faster than BM25.

## 5. Preliminary versus final 10k results

The first 10k attempt with only 10 timed searches reported **30.945 ms mean**, **125.574 ms p95**, and **213.946 ms maximum** for semantic search. A subsequent, larger 100-query run reported **10.848 ms mean**, **11.397 ms p95**, and **12.048 ms maximum**.

This supports treating the initial high-tail result as an unstable preliminary measurement. It does **not** identify the root cause of the earlier outliers. More repetitions under controlled load would strengthen claims about tail latency.

In [ ]:
rows=[]
for name,label in [('10k_initial','Preliminary 10k · 10 runs'), ('10k_final','Final 10k · 100 runs')]:
    b=REPORTS[name]
    for engine in ['semantic','bm25']:
        z=b[engine]
        rows.append({'Run':label,'Engine':engine.upper(),'Mean ms':z['mean'], 'p50 ms':z['p50'],
                     'p95 ms':z['p95'], 'p99 ms':z['p99'], 'Max ms':z['max'], 'Samples':b['timed_queries']})
display(pd.DataFrame(rows))

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.5))
names=['Initial 10k (n=10)', 'Final 10k (n=100)']
ax.bar(names,[REPORTS['10k_initial']['semantic']['p95'], REPORTS['10k_final']['semantic']['p95']])
ax.axhline(100,ls='--',label='100 ms target')
ax.set(ylabel='Semantic p95 (ms)',title='Semantic p95: preliminary vs final benchmark')
ax.legend();ax.grid(axis='y',alpha=.2)
plt.tight_layout();plt.show()

## 6. 1k versus 10k: descriptive scaling comparison

The earlier 1k run had 30 queries, whereas the final 10k run used 100 queries. Results are shown for context, but **different query counts and potentially different environmental conditions prevent a controlled scaling inference**. For example, the earlier 1k semantic mean (28.25 ms) exceeds the final 10k mean (10.848 ms); that is not evidence that larger indexes inherently accelerate search.

In [ ]:
scaling=[]
for key,label in [('1k_previous','1,000 (earlier)'),('10k_final','10,000 (final)')]:
    r=REPORTS[key]
    scaling.append({'Corpus':label,'Timed queries':r['timed_queries'],
       'Semantic mean (ms)':r['semantic']['mean'], 'Semantic p95 (ms)':r['semantic']['p95'],
       'BM25 mean (ms)':r['bm25']['mean'],'BM25 p95 (ms)':r['bm25']['p95']})
display(pd.DataFrame(scaling))

## 7. Offline indexing and model-load costs

In [ ]:
build = pd.DataFrame([
  {'Run':'Preliminary 10k','Model loading (s)':8.594,'Embeddings + FAISS (s)':123.974,'BM25 build (s)':1.040},
  {'Run':'Final 10k','Model loading (s)':8.844,'Embeddings + FAISS (s)':136.250,'BM25 build (s)':1.054},
])
display(build)
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(['Model load','Embeddings + FAISS','BM25 build'],
       [final['model_load_s'],final['faiss_build_s'],final['bm25_build_s']])
ax.set(ylabel='Seconds', title='Final 10k offline preparation time')
ax.grid(axis='y',alpha=.2)
plt.tight_layout();plt.show()
print(f"FAISS+embedding indexing per document (average): {1000*final['faiss_build_s']/10000:.2f} ms/listing")

Indexing was approximately **136.25 seconds** for embedding generation plus FAISS construction in the final run; this is a one-time or batch-update cost, not per-query latency. BM25 construction took approximately **1.054 seconds**. Production systems normally persist the dense index and update it as new listings arrive.

## 8. Relevance-judgment dataset

A separate annotation pass assigned 0/1/2 relevance grades to **50 query–listing pairs**. The judging rubric was:

- **2 — Highly relevant:** Explicit evidence that the listing satisfies the query.
- **1 — Partially relevant:** Related information, or incomplete evidence for full satisfaction.
- **0 — Not relevant or unverified:** The available listing description does not establish that the requested conditions are met.

The label-0 bucket mixes negative evidence with **unknown** cases (notably missing asking prices). For rigorous ranking evaluation, consider splitting unknown judgments from explicit nonrelevance and using actual structured listing fields for price conditions.

In [ ]:
# This labeled file was generated in the Week 5 annotation step.
# It is optional when sharing the notebook separately from the supplied CSV.
paths = [
    Path('data/processed/relevance_judgments.csv'),
    Path('data/processed/relevance_judgments_labeled.csv'),
    Path('relevance_judgments_labeled.csv'),
    Path('/mnt/data/relevance_judgments_labeled.csv'),
]
found = next((p for p in paths if p.exists()), None)
if found is not None:
    judgments = pd.read_csv(found)
    counts = judgments['relevance'].value_counts(dropna=False).sort_index()
    print('Loaded:', found)
    print('Judged pairs:', len(judgments))
    print('Distinct queries:', judgments['query'].nunique())
    print('Relevance distribution:', counts.to_dict())
else:
    judgments = None
    counts = pd.Series({0:4, 1:4, 2:42})
    print('Labeled CSV not found; showing supplied aggregate counts instead.')
display(pd.DataFrame({'Relevance grade': [0,1,2],
  'Count': [int(counts.get(i,0)) for i in [0,1,2]]}))
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(['0 · Not relevant / unverified','1 · Partial','2 · High'],
       [int(counts.get(i,0)) for i in [0,1,2]])
ax.set(ylabel='Judged pairs',title='Relevance-label distribution (50 pairs)')
plt.xticks(rotation=12,ha='right');plt.tight_layout();plt.show()

### Why full retrieval-quality metrics are still pending

These 50 judgments correspond to **50 distinct queries**, with **one assessed result per query**. You cannot treat every unjudged top-10 search result as nonrelevant. Therefore, results such as **Precision@10, Recall@10, MAP, and NDCG@10 for FAISS versus BM25 are not reported** from this dataset. The relevance distribution is an annotation summary, not a search-system quality score. The separate latency benchmark does not depend on relevance labels.

## 9. How to complete the FAISS vs BM25 relevance comparison

1. Take a representative query set (the existing 50 are a useful start).
2. Retrieve each query's top 10 from **both engines**, deduplicate by `(query, listing_id)`, and retain the source-engine ranks.
3. Add random negatives or hard negatives where useful; judge the **full listing and structured price/bed/bath fields** rather than only truncated excerpts.
4. Assign independent 0/1/2 grades, including an **unjudged/unknown** state when evidence is insufficient.
5. Compute metrics using consistent denominators and clearly documented handling of unjudged items. For graded relevance, use NDCG@10; for binary relevance use Precision@10 / Recall@10 if reasonably complete judgments are available.
6. Report per-query differences as well as corpus averages to identify queries where semantic similarity or exact keywords are more effective.

Avoid circular judgments generated by the same retriever being evaluated.

In [ ]:
# Optional: inspect data coverage by query if the labeled CSV is available.
if judgments is not None:
    judgment_counts = judgments.groupby('query', dropna=False).size()
    print('Minimum judgments/query:', judgment_counts.min())
    print('Maximum judgments/query:', judgment_counts.max())
    print('Queries with >= 10 judgments:', int((judgment_counts >= 10).sum()))
else:
    print('Run with the labeled CSV available to inspect query coverage.')

## 10. Reproducing the 10k benchmark locally

Use the updated `scripts/evaluate_search.py` from Week 5 and run from the repository root. The CSV must contain 10,000 valid distinct listing IDs. The benchmark loads the model and builds the index once, warms both engines, and times `search` calls for 100 queries.

```bash
export TOKENIZERS_PARALLELISM=false
export OMP_NUM_THREADS=1
export MKL_NUM_THREADS=1
export OPENBLAS_NUM_THREADS=1
export VECLIB_MAXIMUM_THREADS=1

python3 -X faulthandler scripts/evaluate_search.py \
  --csv data/processed/listing_sample_10k.csv \
  --target 10000 --warmup 10 --runs 100 \
  --report reports/search_comparison_10k.json
```

Capture the Mac hardware, OS, Python, Torch, Sentence Transformers, FAISS, and NumPy versions for reproducibility. Check that the same top-10 retrieval and query mix are used across runs. Run the benchmark several times for stable p95/p99 estimates.

In [ ]:
# Optional: compare saved benchmark report to the quoted final measurements.
report_path = Path('reports/search_comparison_10k.json')
if report_path.exists():
    actual = json.loads(report_path.read_text())
    print('Saved report found:', report_path)
    print('Listing count:', actual.get('listing_count'))
    print('Reported semantic latency:', actual.get('latency_ms', {}).get('semantic'))
else:
    print('No local report JSON found. The report uses the final supplied measurements above.')

## 11. Discussion, limitations, and final conclusion

**Findings.** A 384-dimensional MiniLM + FAISS search implementation handled **10,000 distinct real-estate listings**. The final warm end-to-end semantic search latency was **10.848 ms mean, 11.001 ms median, 11.397 ms p95, 11.612 ms p99, and 12.048 ms maximum** over 100 timed queries. BM25 achieved **14.686 ms mean and 18.352 ms p95**. Both engines satisfied the 100 ms benchmark threshold under the measured conditions. Offline embedding generation and FAISS construction took **136.250 seconds**, while BM25 construction took **1.054 seconds**.

**Interpretation.** The target of **under 100 ms query latency at 10k listings is met** for mean and p95 in the supplied final results. The 10-query preliminary run showed much worse semantic tail latency, so the final run should be treated as the principal result, with reproducibility over multiple sessions recommended.

**Limitations.** Results were recorded on a particular local machine and workload, with no hardware specifications or repeated-session variance recorded. FAISS `IndexFlatIP` is exhaustive rather than approximate indexing at this scale; larger corpora or stricter memory budgets may favor other FAISS indexes. Lexical BM25 and dense semantic scores are not directly numerically comparable. The labeled 50-pair dataset is insufficient for fair ranking-quality comparisons at top 10. No relevance superiority of one engine over the other is claimed.

**Follow-up work.** Pool results from both engines, expand human judgments, compute NDCG@10 and Precision@10, and test hybrid retrieval with the Week 4 structured query parser for constraints such as price, bedrooms, bathrooms, and city.

## Appendix — Source and provenance of numbers

- The **final 10,000-listing metrics** come directly from the benchmark JSON supplied in this conversation (100 timed queries, 10 warm-ups, top_k=10).
- The **preliminary 10,000-listing metrics** come from the earlier JSON supplied in this conversation (10 timed queries, 2 warm-ups).
- The **1,000-listing metrics** come from the previous Week 5 latency JSON (30 runs).
- The **50 graded relevance pairs** come from the labeled `relevance_judgments_labeled.csv` file created earlier (42 grade 2, 4 grade 1, 4 grade 0).
- Charts reproduce these reported values; the notebook does **not** manufacture additional latency observations or infer retrieval metrics that were not computed.